<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/New_vidarabine_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 42.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_vidarabine_deduplication(sdf_input="Vidarabine_Similar.sdf", csv_input="Vidarabine_Similar.csv"):
    """
    Strictly handles Deduplication (Step 1) by ingesting the raw SDF,
    generating canonical descriptors, and removing identical structures.
    """
    print(f"🧬 Initializing analog deduplication routine on target file: '{sdf_input}'...")

    # Load raw molecules directly from your uploaded SDF file
    try:
        supplier = Chem.SDMolSupplier(sdf_input, sanitize=True, removeHs=True)
    except Exception:
        print(f"❌ Failed to parse '{sdf_input}'. Check your sidebar file spelling.")
        return None

    # Connect to the metadata sheet to help parse original data rows
    try:
        df_metadata = pd.read_csv(csv_input)
        print(f"📊 Connected to metadata sheet containing {len(df_metadata)} data rows.\n")
    except Exception:
        print(f"⚠️ Metadata sheet '{csv_input}' not found. Processing SDF entries only.\n")

    seen_inchikeys = set()
    deduplicated_records = []
    duplicate_count = 0

    print(f"🔄 Auditing structural keys across raw data entries...\n")

    for i, mol in enumerate(supplier):
        if mol is None:
            continue

        # Pull original compound ID if it exists, otherwise generate one
        cid = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Vidarabine_Analog_{i}"

        # 1. Generate Canonical Isomeric SMILES and InChIKey (Your exact logic)
        try:
            canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
            inchikey = Chem.MolToInchiKey(mol)
        except Exception:
            continue

        # 2. Deduplicate based on InChIKey hash values
        if inchikey in seen_inchikeys:
            duplicate_count += 1
            continue

        seen_inchikeys.add(inchikey)

        deduplicated_records.append({
            'Compound_ID': f"CID_{cid}" if not str(cid).startswith("CID") else cid,
            'SMILES': canonical_smiles,
            'InChIKey_Hash': inchikey
        })

    # Compile into clean operational base frame
    df_clean_base = pd.DataFrame(deduplicated_records)

    # Save the master deduplicated 2D dataset to your workspace
    output_csv = "Vidarabine_Analogs_Clean_Base_2D.csv"
    df_clean_base.to_csv(output_csv, index=False)

    print("✨ Vidarabine Deduplication Subroutine Complete!")
    print(f"📊 Total raw entries processed from SDF: {len(supplier)}")
    print(f"✂️ Redundant duplicate records discarded: {duplicate_count}")
    print(f"🎯 Total unique analog leads preserved: {len(df_clean_base)}")
    print(f"📁 Deduplicated base collection exported to: '{output_csv}'")

    return df_clean_base

# --- Run the Deduplication Sequence ---
df_clean_base = run_vidarabine_deduplication()

# Display the interactive deduplicated database
df_clean_base

🧬 Initializing analog deduplication routine on target file: 'Vidarabine_Similar.sdf'...
❌ Failed to parse 'Vidarabine_Similar.sdf'. Check your sidebar file spelling.


In [ ]:
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_remdesivir_deduplication(sdf_input="Remdesivir_Similar.sdf", csv_input="Remdesivir_Similar.csv"):
    """
    Strictly handles Deduplication (Step 1) for the Remdesivir analog library.
    Uses an explicit stream block to seamlessly bypass platform parsing limitations.
    """
    print(f"🧬 Initializing analog deduplication routine on target file: '{sdf_input}'...")

    seen_inchikeys = set()
    deduplicated_records = []
    duplicate_count = 0
    raw_processed_count = 0

    print(f"🔄 Streaming and auditing structural keys across raw data entries...\n")

    # Use an explicit binary file stream block to feed the supplier safely line-by-line
    try:
        with open(sdf_input, 'rb') as f:
            supplier = Chem.ForwardSDMolSupplier(f, sanitize=True, removeHs=True)

            for mol in supplier:
                if mol is None:
                    continue

                raw_processed_count += 1

                # Pull original compound ID if it exists, otherwise generate one
                cid = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Remdesivir_Analog_{raw_processed_count}"

                # Generate Canonical Isomeric SMILES and InChIKey
                try:
                    canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
                    inchikey = Chem.MolToInchiKey(mol)
                except Exception:
                    continue

                # Deduplicate based on exact stereospecific InChIKey hash values
                if inchikey in seen_inchikeys:
                    duplicate_count += 1
                    continue

                seen_inchikeys.add(inchikey)

                deduplicated_records.append({
                    'Compound_ID': f"CID_{cid}" if not str(cid).startswith("CID") else cid,
                    'SMILES': canonical_smiles,
                    'InChIKey_Hash': inchikey
                })
    except Exception as e:
        print(f"❌ Critical Error reading file: {str(e)}")
        return None

    # Compile into clean operational base frame
    df_clean_base = pd.DataFrame(deduplicated_records)

    # Save the master deduplicated 2D dataset to your workspace
    output_csv = "Remdesivir_Analogs_Clean_Base_2D.csv"
    df_clean_base.to_csv(output_csv, index=False)

    print("✨ Remdesivir Deduplication Subroutine Complete!")
    print(f"📊 Total raw entries processed from SDF: {raw_processed_count}")
    print(f"✂️ Redundant duplicate records discarded: {duplicate_count}")
    print(f"🎯 Total unique analog leads preserved: {len(df_clean_base)}")
    print(f"📁 Deduplicated base collection exported to: '{output_csv}'")

    return df_clean_base

# --- Run the Deduplication Sequence ---
df_clean_base = run_remdesivir_deduplication()

# Display the interactive deduplicated database
df_clean_base

🧬 Initializing analog deduplication routine on target file: 'Remdesivir_Similar.sdf'...
🔄 Streaming and auditing structural keys across raw data entries...

✨ Remdesivir Deduplication Subroutine Complete!
📊 Total raw entries processed from SDF: 791
✂️ Redundant duplicate records discarded: 0
🎯 Total unique analog leads preserved: 791
📁 Deduplicated base collection exported to: 'Remdesivir_Analogs_Clean_Base_2D.csv'


,Compound_ID,SMILES,InChIKey_Hash
0,CID_121304016,CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@]...,RWWYLEGWBNMMLJ-YSOARWBDSA-N
1,CID_58059494,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,YAAQYJCOIFNMKX-CVANIGNKSA-N
2,CID_56832851,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-MEUHYHILSA-N
3,CID_54759694,CC(C)OC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C#...,YAAQYJCOIFNMKX-RSTNYOGXSA-N
4,CID_44468358,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,GELKCCYSGJABSF-YQLZXLBKSA-N
...,...,...,...
786,CID_176484592,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@@...,VQQVLPGGEPQOSU-ZRVDAUNESA-N
787,CID_176484593,CCC(C)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@@]...,HKKRLUUBLKWCBO-WOVFFQEMSA-N
788,CID_177678924,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-WEVWXOSDSA-N
789,CID_177828538,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@]...,RWWYLEGWBNMMLJ-CSVWCILCSA-N


In [ ]:
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Maintain interactive spreadsheet layout in Colab
data_table.enable_dataframe_formatter()

def run_production_remdesivir_sanitization(input_csv="Remdesivir_Analogs_Clean_Base_2D.csv"):
    """
    Ingests the deduplicated Remdesivir library, validates atomic valences,
    standardizes core ring aromaticity, and purges unphysical compounds.
    """
    print(f"🧬 Loading unique analog database: '{input_csv}'...")
    try:
        df_base = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Ensure Step 1 completed cleanly.")
        return None

    print(f"⚡ Processing {len(df_base)} compounds through structural stabilization loops...\n")

    sanitized_records = []
    history_log = []

    verified_count = 0
    discarded_count = 0

    for idx, row in df_base.iterrows():
        comp_id = row['Compound_ID']
        raw_smiles = row['SMILES']
        old_inchikey = row['InChIKey_Hash']

        mol = Chem.MolFromSmiles(raw_smiles)

        if mol is not None:
            try:
                # 1. Standardize aromatic systems and fix basic valence bugs
                Chem.SanitizeMol(mol)
                # 2. Perform general electronic structure cleanups
                Chem.Cleanup(mol)

                # Regenerate pristine canonical strings post-sanitization
                clean_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
                clean_inchikey = Chem.MolToInchiKey(mol)

                verified_count += 1
                status = "Verified"

                sanitized_records.append({
                    'Compound_ID': comp_id,
                    'SMILES': clean_smiles,
                    'InChIKey_Hash': clean_inchikey
                })

            except Exception as e:
                verified_count += 0 # Failed validation step
                discarded_count += 1
                status = f"Failed Sanitization: {str(e)}"
                clean_smiles = "Sanitization Failed"
                clean_inchikey = "N/A"
        else:
            discarded_count += 1
            status = "Invalid SMILES"
            clean_smiles = "Parsing Failed"
            clean_inchikey = "N/A"

        # Track history comparisons (Your dual-table log style logic)
        history_log.append({
            'Compound_ID': comp_id,
            'Stage': '1_Deduplicated',
            'SMILES': raw_smiles,
            'InChIKey': old_inchikey,
            'Audit_Status': 'Pass'
        })
        history_log.append({
            'Compound_ID': comp_id,
            'Stage': '2_Sanitized',
            'SMILES': clean_smiles,
            'InChIKey': clean_inchikey,
            'Audit_Status': status
        })

    # Generate output datasets
    df_sanitized_master = pd.DataFrame(sanitized_records)
    df_history_report = pd.DataFrame(history_log)

    # Save a permanent step-by-step optimization record to disk
    df_history_report.to_csv("Remdesivir_Analogs_Sanitization_Audit.csv", index=False)

    # Export the clean master library for the next step
    output_master_csv = "Remdesivir_Analogs_Sanitized_Master_2D.csv"
    df_sanitized_master.to_csv(output_master_csv, index=False)

    print("✨ High-Throughput Sanitization & Valence Audit Complete!")
    print(f"🟩 Structurally verified chemical structures: {verified_count}")
    print(f"🟥 Structurally impossible compounds discarded: {discarded_count}")
    print(f"📁 Step-by-step history log saved as: 'Remdesivir_Analogs_Sanitization_Audit.csv'")
    print(f"📁 Clean master library ready for next phase saved as: '{output_master_csv}'")

    return df_history_report

# --- Execute Electronic Audit Pipeline ---
df_sanitization_report = run_production_remdesivir_sanitization()

# View interactive history grid matching your dual-row display style
df_sanitization_report

🧬 Loading unique analog database: 'Remdesivir_Analogs_Clean_Base_2D.csv'...
⚡ Processing 791 compounds through structural stabilization loops...

✨ High-Throughput Sanitization & Valence Audit Complete!
🟩 Structurally verified chemical structures: 791
🟥 Structurally impossible compounds discarded: 0
📁 Step-by-step history log saved as: 'Remdesivir_Analogs_Sanitization_Audit.csv'
📁 Clean master library ready for next phase saved as: 'Remdesivir_Analogs_Sanitized_Master_2D.csv'


,Compound_ID,Stage,SMILES,InChIKey,Audit_Status
0,CID_121304016,1_Deduplicated,CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@]...,RWWYLEGWBNMMLJ-YSOARWBDSA-N,Pass
1,CID_121304016,2_Sanitized,CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@]...,RWWYLEGWBNMMLJ-YSOARWBDSA-N,Verified
2,CID_58059494,1_Deduplicated,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,YAAQYJCOIFNMKX-CVANIGNKSA-N,Pass
3,CID_58059494,2_Sanitized,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,YAAQYJCOIFNMKX-CVANIGNKSA-N,Verified
4,CID_56832851,1_Deduplicated,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-MEUHYHILSA-N,Pass
...,...,...,...,...,...
1577,CID_177678924,2_Sanitized,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-WEVWXOSDSA-N,Verified
1578,CID_177828538,1_Deduplicated,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@]...,RWWYLEGWBNMMLJ-CSVWCILCSA-N,Pass
1579,CID_177828538,2_Sanitized,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@]...,RWWYLEGWBNMMLJ-CSVWCILCSA-N,Verified
1580,CID_177834663,1_Deduplicated,CCC(CC)COC(=O)[C@@H](C)N[P@@](=O)(OC[C@H]1O[C@...,RWWYLEGWBNMMLJ-IDNGIRRQSA-N,Pass


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import SaltRemover
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def remove_salts_and_fragments(smiles):
    """
    Removes salts/counter-ions and retains the largest covalent fragment.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        # Initialize the RDKit SaltRemover
        remover = SaltRemover.SaltRemover()

        # Strip away salts and keep only the active drug core molecule
        cleansed_mol = remover.StripMol(mol, dontRemoveEverything=True)

        # In case of multiple separate fragments, keep only the largest one
        frags = Chem.GetMolFrags(cleansed_mol, asMols=True)
        if frags:
            largest_frag = max(frags, default=cleansed_mol, key=lambda m: m.GetNumAtoms())
            return Chem.MolToSmiles(largest_frag, isomericSmiles=True)
    return smiles

def run_production_remdesivir_desalting(input_csv="Remdesivir_Analogs_Sanitized_Master_2D.csv"):
    """
    Loads sanitized entries, applies desalting, and builds the step-by-step
    comparison table showing the transformation.
    """
    print(f"🧬 Loading sanitized database: '{input_csv}'...")
    try:
        df_sanitized = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Make sure Step 2 ran completely.")
        return None

    print(f"🧂 Commencing fragment stripping on {len(df_sanitized)} compounds...\n")

    rows = []
    desalted_records = []
    multi_fragment_count = 0

    for idx, row in df_sanitized.iterrows():
        comp_id = row['Compound_ID']
        sanitized_smiles = row['SMILES']
        orig_inchikey = row['InChIKey_Hash']

        # Apply the salt and fragment removal routine
        desalted_smiles = remove_salts_and_fragments(sanitized_smiles)

        # Regenerate InChIKey post-desalting
        try:
            desalted_mol = Chem.MolFromSmiles(desalted_smiles)
            desalted_inchikey = Chem.MolToInchiKey(desalted_mol)
        except Exception:
            desalted_inchikey = "N/A"

        # Track if a compound was actually split/altered
        if desalted_smiles != sanitized_smiles:
            multi_fragment_count += 1

        # Row 1: The Sanitized State (Pre-desalting)
        rows.append({
            'Compound_ID': comp_id,
            'process_step': 'Sanitization',
            'result_smiles': sanitized_smiles,
            'inchikey': orig_inchikey
        })

        # Row 2: The Desalted State (Post-desalting)
        rows.append({
            'Compound_ID': comp_id,
            'process_step': 'Desalting',
            'result_smiles': desalted_smiles,
            'inchikey': desalted_inchikey
        })

        # Save to master list for saving to disk
        desalted_records.append({
            'Compound_ID': comp_id,
            'SMILES': desalted_smiles,
            'InChIKey_Hash': desalted_inchikey
        })

    # Create the comparison table matching your style preference
    df_comparison = pd.DataFrame(rows)

    # Export the pure, clean 2D library for the next step
    df_master_clean = pd.DataFrame(desalted_records)
    output_csv = "Remdesivir_Analogs_Desalted_Master_2D.csv"
    df_master_clean.to_csv(output_csv, index=False)

    print("✨ High-Throughput Desalting Complete!")
    print(f"🟩 Total compounds successfully evaluated: {len(df_sanitized)}")
    print(f"✂️ Multi-fragment salt systems uncoupled: {multi_fragment_count}")
    print(f"📁 Pure parent ligand library exported to: '{output_csv}'")

    return df_comparison

# --- Execute Desalting Only ---
df_comparison = run_production_remdesivir_desalting()

# Display the interactive comparison matrix
df_comparison

🧬 Loading sanitized database: 'Remdesivir_Analogs_Sanitized_Master_2D.csv'...
🧂 Commencing fragment stripping on 791 compounds...

✨ High-Throughput Desalting Complete!
🟩 Total compounds successfully evaluated: 791
✂️ Multi-fragment salt systems uncoupled: 7
📁 Pure parent ligand library exported to: 'Remdesivir_Analogs_Desalted_Master_2D.csv'


,Compound_ID,process_step,result_smiles,inchikey
0,CID_121304016,Sanitization,CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@]...,RWWYLEGWBNMMLJ-YSOARWBDSA-N
1,CID_121304016,Desalting,CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@]...,RWWYLEGWBNMMLJ-YSOARWBDSA-N
2,CID_58059494,Sanitization,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,YAAQYJCOIFNMKX-CVANIGNKSA-N
3,CID_58059494,Desalting,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,YAAQYJCOIFNMKX-CVANIGNKSA-N
4,CID_56832851,Sanitization,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-MEUHYHILSA-N
...,...,...,...,...
1577,CID_177678924,Desalting,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-WEVWXOSDSA-N
1578,CID_177828538,Sanitization,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@]...,RWWYLEGWBNMMLJ-CSVWCILCSA-N
1579,CID_177828538,Desalting,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@]...,RWWYLEGWBNMMLJ-CSVWCILCSA-N
1580,CID_177834663,Sanitization,CCC(CC)COC(=O)[C@@H](C)N[P@@](=O)(OC[C@H]1O[C@...,RWWYLEGWBNMMLJ-IDNGIRRQSA-N


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from google.colab import data_table

# Keep Colab spreadsheet layouts interactive
data_table.enable_dataframe_formatter()

def standardize_functional_groups(smiles):
    """
    Normalizes structural functional domains and charge representations
    using the official RDKit standardizer engine.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        try:
            # Apply RDKit's high-level standardization protocol
            standardized_mol = rdMolStandardize.StandardizeMol(mol)
            return Chem.MolToSmiles(standardized_mol, isomericSmiles=True)
        except Exception:
            return smiles
    return None

def run_production_remdesivir_standardization(input_csv="Remdesivir_Analogs_Desalted_Master_2D.csv"):
    """
    Ingests desalted master data, executes functional resonance standardization,
    and returns a dual-state history tracking table matching your exact process layout.
    """
    print(f"🧬 Loading desalted master library: '{input_csv}'...")
    try:
        df_desalted = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Verify Step 3 ran successfully.")
        return None

    print(f"⚖️ Harmonizing functional groups across {len(df_desalted)} unique compounds...\n")

    rows = []
    standardized_records = []
    resonance_shifts = 0

    for idx, row in df_desalted.iterrows():
        comp_id = row['Compound_ID']
        desalted_smiles = row['SMILES']
        orig_inchikey = row['InChIKey_Hash']

        # Execute functional group normalization
        std_smiles = standardize_functional_groups(desalted_smiles)

        # Recompute InChIKey post-standardization
        try:
            std_mol = Chem.MolFromSmiles(std_smiles)
            std_inchikey = Chem.MolToInchiKey(std_mol)
        except Exception:
            std_inchikey = "N/A"

        # Track structural modification counts
        if std_inchikey != orig_inchikey:
            resonance_shifts += 1

        # Row 1: The Desalted Base State (Matching your process presentation style)
        rows.append({
            'Compound_ID': comp_id,
            'process_step': 'Desalting',
            'result_smiles': desalted_smiles,
            'inchikey': orig_inchikey
        })

        # Row 2: The Standardized Core State
        rows.append({
            'Compound_ID': comp_id,
            'process_step': 'Standardization',
            'result_smiles': std_smiles,
            'inchikey': std_inchikey
        })

        # Track clean master matrix to save down
        standardized_records.append({
            'Compound_ID': comp_id,
            'SMILES': std_smiles,
            'InChIKey_Hash': std_inchikey
        })

    # Compile into standard tracking structures
    df_comparison = pd.DataFrame(rows)
    df_master_clean = pd.DataFrame(standardized_records)

    # Save the standardized 2D chemical dataset to your environment disk
    output_csv = "Remdesivir_Analogs_Standardized_Master_2D.csv"
    df_master_clean.to_csv(output_csv, index=False)

    print("✨ High-Throughput Functional Group Standardization Complete!")
    print(f"🟩 Total analog files audited: {len(df_desalted)}")
    print(f"⚖️ Resonance frameworks/charges explicitly normalized: {resonance_shifts}")
    print(f"📁 Standardized master data collection exported to: '{output_csv}'")

    return df_comparison

# --- Execute Standardization ---
df_comparison = run_production_remdesivir_standardization()

# Display interactive process layout matrix
df_comparison

🧬 Loading desalted master library: 'Remdesivir_Analogs_Desalted_Master_2D.csv'...
⚖️ Harmonizing functional groups across 791 unique compounds...

✨ High-Throughput Functional Group Standardization Complete!
🟩 Total analog files audited: 791
⚖️ Resonance frameworks/charges explicitly normalized: 0
📁 Standardized master data collection exported to: 'Remdesivir_Analogs_Standardized_Master_2D.csv'


,Compound_ID,process_step,result_smiles,inchikey
0,CID_121304016,Desalting,CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@]...,RWWYLEGWBNMMLJ-YSOARWBDSA-N
1,CID_121304016,Standardization,CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@]...,RWWYLEGWBNMMLJ-YSOARWBDSA-N
2,CID_58059494,Desalting,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,YAAQYJCOIFNMKX-CVANIGNKSA-N
3,CID_58059494,Standardization,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,YAAQYJCOIFNMKX-CVANIGNKSA-N
4,CID_56832851,Desalting,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-MEUHYHILSA-N
...,...,...,...,...
1577,CID_177678924,Standardization,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-WEVWXOSDSA-N
1578,CID_177828538,Desalting,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@]...,RWWYLEGWBNMMLJ-CSVWCILCSA-N
1579,CID_177828538,Standardization,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@]...,RWWYLEGWBNMMLJ-CSVWCILCSA-N
1580,CID_177834663,Desalting,CCC(CC)COC(=O)[C@@H](C)N[P@@](=O)(OC[C@H]1O[C@...,RWWYLEGWBNMMLJ-IDNGIRRQSA-N


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from google.colab import data_table

# Enable interactive table display inside Colab
data_table.enable_dataframe_formatter()

def calculate_drug_likeness(smiles):
    """
    Calculates detailed Lipinski and Veber molecular descriptors.
    """
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return pd.Series([None] * 8)

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rotb = Lipinski.NumRotatableBonds(mol)

    # Check traditional Lipinski rules strictly
    lipinski_violations = sum([mw > 600, logp > 5, hbd > 5, hba > 10])

    # Check traditional Veber parameters
    veber_violations = sum([tpsa > 140, rotb > 10])

    return pd.Series([mw, logp, hbd, hba, tpsa, rotb, lipinski_violations, veber_violations])

def run_production_remdesivir_profiling(input_csv="Remdesivir_Analogs_Standardized_Master_2D.csv"):
    """
    Loads your standardized Remdesivir database, profiles descriptors,
    and isolates viable configurations using adapted prodrug space filters.
    """
    print(f"🧬 Loading standardized master database: '{input_csv}'...")
    try:
        df_std = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Verify Step 4 ran successfully.")
        return None

    print(f"📊 Running descriptor math across {len(df_std)} prioritized analogs...\n")

    # Clone base structures and apply descriptor computations
    df_profile = df_std.copy()
    cols = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotB', 'Lipinski_Violations', 'Veber_Violations']

    # Apply calculations to the SMILES column
    df_profile[cols] = df_profile['SMILES'].apply(calculate_drug_likeness)

    # Apply tailored prodrug constraints (MW <= 600, LogP <= 6.5, Rotatable Bonds <= 15)
    df_profile['Passed_Prodrug_Filter'] = df_profile.apply(
        lambda x: "Yes" if (x['MW'] <= 600) and (-2.0 <= x['LogP'] <= 6.5) and (x['RotB'] <= 15) else "No",
        axis=1
    )

    # Isolate screened records that pass the structural profile filter
    df_screened_master = df_profile[df_profile['Passed_Prodrug_Filter'] == "Yes"].copy()

    # Clean up column configurations for final export
    df_screened_master = df_screened_master.drop(columns=['Passed_Prodrug_Filter'])

    # Save output repositories directly to workspace disk
    df_profile.to_csv("Remdesivir_Analogs_Full_Drug_Likeness_Profiles.csv", index=False)

    output_master_csv = "Remdesivir_Analogs_Profileed_Master_2D.csv"
    df_screened_master[['Compound_ID', 'SMILES', 'InChIKey_Hash']].to_csv(output_master_csv, index=False)

    passed_count = len(df_screened_master)
    rejected_count = len(df_profile) - passed_count

    print("✨ High-Throughput Drug-Likeness Profiling Complete!")
    print(f"🟩 Analogs preserved within target prodrug space: {passed_count}")
    print(f"🟥 Extreme structural outliers rejected: {rejected_count}")
    print(f"📁 Detailed molecular parameters sheet saved as: 'Remdesivir_Analogs_Full_Drug_Likeness_Profiles.csv'")
    print(f"📁 Profileed collection ready for next phase saved as: '{output_master_csv}'")

    return df_profile

# --- Run Profiling Pipeline Subroutine ---
df_drug_likeness = run_production_remdesivir_profiling()

# View interactive properties chart mapping exactly to your workspace output variable
df_drug_likeness

🧬 Loading standardized master database: 'Remdesivir_Analogs_Standardized_Master_2D.csv'...
📊 Running descriptor math across 791 prioritized analogs...

✨ High-Throughput Drug-Likeness Profiling Complete!
🟩 Analogs preserved within target prodrug space: 301
🟥 Extreme structural outliers rejected: 490
📁 Detailed molecular parameters sheet saved as: 'Remdesivir_Analogs_Full_Drug_Likeness_Profiles.csv'
📁 Profileed collection ready for next phase saved as: 'Remdesivir_Analogs_Profileed_Master_2D.csv'


,Compound_ID,SMILES,InChIKey_Hash,MW,LogP,HBD,HBA,TPSA,RotB,Lipinski_Violations,Veber_Violations,Passed_Prodrug_Filter
0,CID_121304016,CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@]...,RWWYLEGWBNMMLJ-YSOARWBDSA-N,602.585,2.31218,4.0,12.0,203.55,13.0,2.0,2.0,No
1,CID_58059494,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,YAAQYJCOIFNMKX-CVANIGNKSA-N,644.622,2.88138,3.0,13.0,209.62,12.0,2.0,2.0,No
2,CID_56832851,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-MEUHYHILSA-N,602.585,2.31218,4.0,12.0,203.55,13.0,2.0,2.0,No
3,CID_54759694,CC(C)OC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C#...,YAAQYJCOIFNMKX-RSTNYOGXSA-N,644.622,2.88138,3.0,13.0,209.62,12.0,2.0,2.0,No
4,CID_44468358,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,GELKCCYSGJABSF-YQLZXLBKSA-N,574.531,1.67448,4.0,12.0,203.55,10.0,1.0,1.0,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...
786,CID_176484592,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@@...,VQQVLPGGEPQOSU-ZRVDAUNESA-N,692.710,4.53668,3.0,12.0,192.55,16.0,2.0,2.0,No
787,CID_176484593,CCC(C)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@@]...,HKKRLUUBLKWCBO-WOVFFQEMSA-N,678.683,4.14658,3.0,12.0,192.55,15.0,2.0,2.0,No
788,CID_177678924,CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#...,RWWYLEGWBNMMLJ-WEVWXOSDSA-N,602.585,2.31218,4.0,12.0,203.55,13.0,2.0,2.0,No
789,CID_177828538,CCC(CC)COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@]...,RWWYLEGWBNMMLJ-CSVWCILCSA-N,602.585,2.31218,4.0,12.0,203.55,13.0,2.0,2.0,No


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from google.colab import data_table

# Maintain interactive table views in Google Colab
data_table.enable_dataframe_formatter()

def refine_for_physiology(smiles):
    """
    Refines molecules for biological pH 7.4 conditions by resetting
    stray partial charges and isolating the dominant stable tautomer.
    """
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return None

    try:
        # Step 1: Neutralize/uncharge structural domains
        unchanger = rdMolStandardize.Uncharger()
        mol = uncharger.uncharge(mol)

        # Step 2: Extract the dominant stable canonical tautomer core
        te = rdMolStandardize.TautomerEnumerator()
        canonical_mol = te.Canonicalize(mol)

        return Chem.MolToSmiles(canonical_mol, isomericSmiles=True)
    except Exception:
        return smiles

def run_production_physiological_refinement(input_csv="Remdesivir_Analogs_Profileed_Master_2D.csv"):
    """
    Ingests the filtered drug-like library, resolves tautomeric variation states,
    and structures the dual-row comparison history report.
    """
    print(f"🧬 Loading filtered drug-like master library: '{input_csv}'...")
    try:
        df_profiled = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not find '{input_csv}'. Verify your previous filtering run saved cleanly.")
        return None

    print(f"⚡ Processing proton tautomer environments across {len(df_profiled)} compounds at pH 7.4...\n")

    history_rows = []
    final_clean_records = []
    tautomer_shifts = 0

    for idx, row in df_profiled.iterrows():
        comp_id = row['Compound_ID']
        profiled_smiles = row['SMILES']
        old_inchikey = row['InChIKey_Hash']

        # Execute the biological standardization loops
        refined_smiles = refine_for_physiology(profiled_smiles)

        # Build fresh absolute structural hashes
        try:
            refined_mol = Chem.MolFromSmiles(refined_smiles)
            refined_inchikey = Chem.MolToInchiKey(refined_mol)
        except Exception:
            refined_inchikey = "N/A"

        # Count if the core changed tautomeric orientation forms
        if refined_inchikey != old_inchikey:
            tautomer_shifts += 1

        # Row 1: The Profiled/Filtered Entry State
        history_rows.append({
            'Compound_ID': comp_id,
            'process_step': 'Profiling_Filter',
            'result_smiles': profiled_smiles,
            'inchikey': old_inchikey
        })

        # Row 2: The Physiologically Refined Tautomer State
        history_rows.append({
            'Compound_ID': comp_id,
            'process_step': 'Tautomer_Refinement',
            'result_smiles': refined_smiles,
            'inchikey': refined_inchikey
        })

        # Keep for the production ready 2D master export
        final_clean_records.append({
            'Compound_ID': comp_id,
            'SMILES': refined_smiles,
            'InChIKey_Hash': refined_inchikey
        })

    # Convert to presentation structures
    df_comparison = pd.DataFrame(history_rows)
    df_master_final = pd.DataFrame(final_clean_records)

    # Write the permanent 2D endpoint master to your local disk environment
    output_csv = "Remdesivir_Analogs_Final_2D_Master.csv"
    df_master_final.to_csv(output_csv, index=False)

    print("✨ Physiological Tautomerization Subroutine Complete!")
    print(f"🟩 Total structures successfully evaluated and locked at pH 7.4: {len(df_profiled)}")
    print(f"🔄 Shifted/optimized proton tautomers: {tautomer_shifts}")
    print(f"📁 Definitive 2D master collection saved to disk workspace as: '{output_csv}'")

    return df_comparison

# --- Run Tautomer Refinement Pipeline Sequence ---
df_comparison = run_production_physiological_refinement()

# View the tracking output history matrix matching your exact layout design
df_comparison

🧬 Loading filtered drug-like master library: 'Remdesivir_Analogs_Profileed_Master_2D.csv'...
⚡ Processing proton tautomer environments across 301 compounds at pH 7.4...

✨ Physiological Tautomerization Subroutine Complete!
🟩 Total structures successfully evaluated and locked at pH 7.4: 301
🔄 Shifted/optimized proton tautomers: 0
📁 Definitive 2D master collection saved to disk workspace as: 'Remdesivir_Analogs_Final_2D_Master.csv'


,Compound_ID,process_step,result_smiles,inchikey
0,CID_44468358,Profiling_Filter,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,GELKCCYSGJABSF-YQLZXLBKSA-N
1,CID_44468358,Tautomer_Refinement,CC(C)OC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(...,GELKCCYSGJABSF-YQLZXLBKSA-N
2,CID_58059537,Profiling_Filter,COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(c2cc...,XKJKPIAHDRNWFI-UYPWRNPRSA-N
3,CID_58059537,Tautomer_Refinement,COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(c2cc...,XKJKPIAHDRNWFI-UYPWRNPRSA-N
4,CID_58059541,Profiling_Filter,CCOC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)(c2c...,AGKHEEIAZSUNMY-KHKPUGPUSA-N
...,...,...,...,...
597,CID_169446183,Tautomer_Refinement,CCCCOC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@](C#N)...,AWYCETMXRCNOIF-DXNQCVLRSA-N
598,CID_171338476,Profiling_Filter,CCCCOC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C#N...,AWYCETMXRCNOIF-VIIJUVLASA-N
599,CID_171338476,Tautomer_Refinement,CCCCOC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C#N...,AWYCETMXRCNOIF-VIIJUVLASA-N
600,CID_172677464,Profiling_Filter,CC(C)COC(=O)[C@H](C)NP(=O)(OC[C@H]1O[C@@](C#N)...,SVRBERDPWIECAJ-OXTDYMLSSA-N


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from google.colab import data_table

# Keep spreadsheet displays interactive in Colab
data_table.enable_dataframe_formatter()

def enumerate_and_verify_stereo(smiles):
    """
    Scans for undefined stereocenters and expands them into explicit variations.
    Setting onlyUnassigned=True prevents tampering with known chiral configurations.
    """
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return []

    # Configure options to only expand centers lacking explicit 2D drawing directions
    opts = StereoEnumerationOptions(tryEmbedding=True, onlyUnassigned=True)
    isomers = list(EnumerateStereoisomers(mol, options=opts))

    return [Chem.MolToSmiles(x, isomericSmiles=True) for x in isomers]

def run_production_stereo_enumeration(input_csv="Remdesivir_Analogs_Final_2D_Master.csv"):
    """
    Ingests the refined pH 7.4 library, maps structural stereocenters, expands
    ambiguous centers, and saves the definitive curated virtual library screen.
    """
    print(f"🧬 Loading biological 2D master library: '{input_csv}'...")
    try:
        df_inputs = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Verify Step 6 executed successfully.")
        return None

    print(f"📐 Auditing stereochemical definitions across {len(df_inputs)} compounds...\n")

    expanded_rows = []
    production_records = []

    total_isomers_generated = 0
    compounds_expanded_count = 0

    for idx, row in df_inputs.iterrows():
        comp_id = row['Compound_ID']
        refined_smiles = row['SMILES']

        # Enumerate unassigned stereocenters
        isomer_list = enumerate_and_verify_stereo(refined_smiles)
        num_isomers = len(isomer_list)

        if num_isomers > 1:
            compounds_expanded_count += 1
        total_isomers_generated += num_isomers

        # Document generated configurations
        for i, iso_smiles in enumerate(isomer_list):
            try:
                iso_mol = Chem.MolFromSmiles(iso_smiles)
                iso_inchikey = Chem.MolToInchiKey(iso_mol)
            except Exception:
                iso_inchikey = "N/A"

            # Create a production tracker identifier index
            isomer_unique_id = f"{comp_id}_iso_{i+1}" if num_isomers > 1 else f"{comp_id}"

            # Append tracking data rows
            expanded_rows.append({
                'Compound_ID': comp_id,
                'Isomer_ID': isomer_unique_id,
                'process_step': 'Stereo_Validation',
                'final_smiles': iso_smiles,
                'inchikey': iso_inchikey
            })

            # Save down metadata properties for the production run master list
            production_records.append({
                'Isomer_ID': isomer_unique_id,
                'Parent_ID': comp_id,
                'SMILES': iso_smiles,
                'InChIKey_Hash': iso_inchikey
            })

    # Convert out to master frames
    df_stereo_report = pd.DataFrame(expanded_rows)
    df_production_library = pd.DataFrame(production_records)

    # Save finalized absolute 2D collections directly to the cloud platform disk
    output_master_csv = "Remdesivir_Analogs_Ready_2D_Screening_Library.csv"
    df_production_library.to_csv(output_master_csv, index=False)

    print("✨ High-Throughput Stereochemical Validation Complete!")
    print(f"🟩 Parent architectures audited: {len(df_inputs)}")
    print(f"🔀 Ambiguous compounds expanded into multiple configurations: {compounds_expanded_count}")
    print(f"🚀 Total verified structures ready for 3D layout: {total_isomers_generated}")
    print(f"📁 Screen-ready 2D master library exported to disk workspace as: '{output_master_csv}'")

    return df_stereo_report

# --- Run Stereo Validation Sequence ---
df_stereo = run_production_stereo_enumeration()

# Display interactive verification matrix
df_stereo

🧬 Loading biological 2D master library: 'Remdesivir_Analogs_Final_2D_Master.csv'...
📐 Auditing stereochemical definitions across 301 compounds...



[14:53:02] UFFTYPER: Warning: hybridization set to SP3 for atom 7
[14:53:02] UFFTYPER: Unrecognized charge state for atom: 7
[14:53:02] UFFTYPER: Warning: hybridization set to SP3 for atom 7
[14:53:02] UFFTYPER: Unrecognized charge state for atom: 7


✨ High-Throughput Stereochemical Validation Complete!
🟩 Parent architectures audited: 301
🔀 Ambiguous compounds expanded into multiple configurations: 224
🚀 Total verified structures ready for 3D layout: 1365
📁 Screen-ready 2D master library exported to disk workspace as: 'Remdesivir_Analogs_Ready_2D_Screening_Library.csv'


,Compound_ID,Isomer_ID,process_step,final_smiles,inchikey
0,CID_44468358,CID_44468358_iso_1,Stereo_Validation,CC(C)OC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C#...,GELKCCYSGJABSF-UZLLJPOSSA-N
1,CID_44468358,CID_44468358_iso_2,Stereo_Validation,CC(C)OC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@@](C...,GELKCCYSGJABSF-FTFXYGJJSA-N
2,CID_58059537,CID_58059537_iso_1,Stereo_Validation,COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C#N)(c...,XKJKPIAHDRNWFI-CNZYZRQUSA-N
3,CID_58059537,CID_58059537_iso_2,Stereo_Validation,COC(=O)[C@H](C)N[P@@](=O)(OC[C@H]1O[C@@](C#N)(...,XKJKPIAHDRNWFI-KITHFANDSA-N
4,CID_58059541,CID_58059541_iso_1,Stereo_Validation,CCOC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C#N)(...,AGKHEEIAZSUNMY-ZUJPLWLLSA-N
...,...,...,...,...,...
1360,CID_169446158,CID_169446158,Stereo_Validation,[2H]C([2H])([2H])[C@]([2H])(N[P@](=O)(OC[C@H]1...,KVHMVOOJHMVWGH-AODUOECFSA-N
1361,CID_169446183,CID_169446183,Stereo_Validation,CCCCOC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@](C#N)...,AWYCETMXRCNOIF-DXNQCVLRSA-N
1362,CID_171338476,CID_171338476,Stereo_Validation,CCCCOC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C#N...,AWYCETMXRCNOIF-VIIJUVLASA-N
1363,CID_172677464,CID_172677464_iso_1,Stereo_Validation,CC(C)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C...,SVRBERDPWIECAJ-VIIJUVLASA-N


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Keep table layouts interactive inside the Colab environment
data_table.enable_dataframe_formatter()

def run_production_3d_generation(input_csv="Remdesivir_Analogs_Ready_2D_Screening_Library.csv", output_sdf="Remdesivir_Analogs_Master_3D.sdf"):
    """
    Ingests the 2D verified screening deck, builds relaxed 3D coordinates
    using ETKDGv3, runs MMFF94 energy minimization, and aggregates all
    successful isomers into a single screening-ready master .sdf file.
    """
    print(f"🧬 Loading validated screening deck: '{input_csv}'...")
    try:
        df_inputs = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Verify your Step 7 run finished cleanly.")
        return None

    print(f"📐 Initializing 3D conformation engine for {len(df_inputs)} structural entries...")
    print(f"📦 All optimized models will combine directly into: '{output_sdf}'\n")

    # Initialize the centralized SDWriter
    sdf_writer = Chem.SDWriter(output_sdf)

    tracking_records = []
    success_count = 0
    failure_count = 0

    # Configure modern ETKDGv3 parameters with a fixed random seed for reproducible geometry
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    # Setting clearConfs=True clears any internal scratchpad shapes to optimize memory usage
    params.clearConfs = True

    for idx, row in df_inputs.iterrows():
        isomer_id = row['Isomer_ID']
        smiles = row['SMILES']

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            tracking_records.append({'Isomer_ID': isomer_id, '3d_status': 'Failed: Invalid SMILES'})
            failure_count += 1
            continue

        try:
            # 1. Add explicit Hydrogens (Crucial for correct 3D volume filling and valence math)
            mol_with_hs = Chem.AddHs(mol)

            # 2. Embed the molecule into realistic 3D coordinates
            embed_status = AllChem.EmbedMolecule(mol_with_hs, params)

            if embed_status == 0:
                # 3. Apply MMFF94 force field optimization to minimize steric strain
                # maxIters=200 provides ample headroom for these specific ring shapes to settle
                AllChem.MMFFOptimizeMolecule(mol_with_hs, maxIters=200)

                # Assign the tracking identifier name properties inside the SDF object entry
                mol_with_hs.SetProp("_Name", isomer_id)

                # Write molecule into the single combined master file
                sdf_writer.write(mol_with_hs)

                tracking_records.append({'Isomer_ID': isomer_id, '3d_status': 'Success'})
                success_count += 1
            else:
                # Fallback check: try embedding with random coordinates if standard distance geometry fails
                params_fallback = AllChem.ETKDGv3()
                params_fallback.randomSeed = 42
                params_fallback.useRandomCoords = True

                embed_status_retry = AllChem.EmbedMolecule(mol_with_hs, params_fallback)
                if embed_status_retry == 0:
                    AllChem.MMFFOptimizeMolecule(mol_with_hs, maxIters=200)
                    mol_with_hs.SetProp("_Name", isomer_id)
                    sdf_writer.write(mol_with_hs)
                    tracking_records.append({'Isomer_ID': isomer_id, '3d_status': 'Success (Random Coords)'})
                    success_count += 1
                else:
                    tracking_records.append({'Isomer_ID': isomer_id, '3d_status': 'Failed: Coordinate embedding error'})
                    failure_count += 1

        except Exception as e:
            tracking_records.append({'Isomer_ID': isomer_id, '3d_status': f'Failed: {str(e)}'})
            failure_count += 1

    # Safely close the combined writer stream
    sdf_writer.close()

    # Create the report tracker dataframe
    df_3d_report = pd.DataFrame(tracking_records)

    print("\n✨ High-Throughput 3D Conformation Generation Complete!")
    print(f"🟩 Successfully optimized and saved structures: {success_count}")
    print(f"🟥 Failed structures: {failure_count}")
    print(f"📁 Master screening file compiled and ready for virtual screening: '{output_sdf}'")

    return df_3d_report

# --- Execute 3D Generation Pipeline ---
df_3d = run_production_3d_generation()

# View interactive embedding summary logs
df_3d

🧬 Loading validated screening deck: 'Remdesivir_Analogs_Ready_2D_Screening_Library.csv'...
📐 Initializing 3D conformation engine for 1365 structural entries...
📦 All optimized models will combine directly into: 'Remdesivir_Analogs_Master_3D.sdf'



[15:03:30] UFFTYPER: Warning: hybridization set to SP3 for atom 7
[15:03:30] UFFTYPER: Unrecognized charge state for atom: 7
[15:03:30] UFFTYPER: Warning: hybridization set to SP3 for atom 7
[15:03:30] UFFTYPER: Unrecognized charge state for atom: 7



✨ High-Throughput 3D Conformation Generation Complete!
🟩 Successfully optimized and saved structures: 1365
🟥 Failed structures: 0
📁 Master screening file compiled and ready for virtual screening: 'Remdesivir_Analogs_Master_3D.sdf'


,Isomer_ID,3d_status
0,CID_44468358_iso_1,Success
1,CID_44468358_iso_2,Success
2,CID_58059537_iso_1,Success
3,CID_58059537_iso_2,Success
4,CID_58059541_iso_1,Success
...,...,...
1360,CID_169446158,Success
1361,CID_169446183,Success
1362,CID_171338476,Success
1363,CID_172677464_iso_1,Success


In [ ]:
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Keep table layouts interactive in your Colab workspace
data_table.enable_dataframe_formatter()

def run_production_3d_qc_and_filter(input_sdf="Remdesivir_Analogs_Master_3D.sdf", output_sdf="Remdesivir_Analogs_Final_Screening_3D.sdf"):
    """
    Streams the master 3D SDF, filters for only primary isomers (_iso_1 or unflagged),
    runs quality control audits on valence/hybridization, and saves the final 3D deck.
    """
    print(f"🧬 Streaming master 3D molecular file: '{input_sdf}'...")

    # Initialize the structure supplier for reading the multi-molecule SDF file
    suppl = Chem.SDMolSupplier(input_sdf, removeHs=False)
    if not suppl:
        print(f"❌ Error: Cannot open '{input_sdf}'. Check your file path.")
        return None

    # Open centralized writer for the production-ready 3D chemical deck
    sdf_writer = Chem.SDWriter(output_sdf)

    qc_records = []
    skipped_isomers = 0
    preserved_count = 0

    print("🔍 Auditing valence parameters and isolating primary screening configs...\n")

    for mol in suppl:
        if mol is None:
            continue

        # Extract the unique compound identifier string assigned in Step 8
        comp_id = mol.GetProp("_Name")

        # FILTER CRITERIA: Keep only primary configurations
        # Skip names containing "_iso_" UNLESS they are the baseline "_iso_1"
        if "_iso_" in comp_id and not comp_id.endswith("_iso_1"):
            skipped_isomers += 1
            continue

        # 1. High-Fidelity Valence Audit
        try:
            mol.UpdatePropertyCache(strict=True)
            valence_status = "Pass"
        except ValueError:
            valence_status = "Fail (Valence Violation)"

        # 2. Extract Geometric Hybridization Metrics
        sp3_count = 0
        sp2_count = 0
        for atom in mol.GetAtoms():
            hyb = atom.GetHybridization()
            if hyb == Chem.rdchem.HybridizationType.SP3:
                sp3_count += 1
            elif hyb == Chem.rdchem.HybridizationType.SP2:
                sp2_count += 1

        # 3. Assess Total Net Charge
        net_charge = Chem.GetFormalCharge(mol)

        # Append audit properties to our tracking report list
        qc_records.append({
            'Compound_ID': comp_id,
            'Valence_Audit': valence_status,
            'SP3_Atom_Count': sp3_count,
            'SP2_Atom_Count': sp2_count,
            'Net_Charge': net_charge
        })

        # Save approved structural records into your deployment screening file
        sdf_writer.write(mol)
        preserved_count += 1

    # Close output stream safely
    sdf_writer.close()

    # Build history layout matrix
    df_qc_report = pd.DataFrame(qc_records)

    # Save a CSV backup log of the quality control check results
    df_qc_report.to_csv("Remdesivir_Analogs_3D_QC_Audit_Log.csv", index=False)

    print("✨ High-Throughput Quality Control Audit Complete!")
    print(f"🟩 Primary screening compounds locked in: {preserved_count}")
    print(f"✂️ Redundant stereoisomeric variants filtered out: {skipped_isomers}")
    print(f"📁 Quality check log table written to: 'Remdesivir_Analogs_3D_QC_Audit_Log.csv'")
    print(f"🚀 PRODUCTION 3D VIRTUAL SCREENING DECK EXPORTED TO: '{output_sdf}'")

    return df_qc_report

# --- Run Quality Control Subroutine ---
df_verify = run_production_3d_qc_and_filter()

# Display interactive validation metrics log
df_verify

🧬 Streaming master 3D molecular file: 'Remdesivir_Analogs_Master_3D.sdf'...
🔍 Auditing valence parameters and isolating primary screening configs...

✨ High-Throughput Quality Control Audit Complete!
🟩 Primary screening compounds locked in: 301
✂️ Redundant stereoisomeric variants filtered out: 1064
📁 Quality check log table written to: 'Remdesivir_Analogs_3D_QC_Audit_Log.csv'
🚀 PRODUCTION 3D VIRTUAL SCREENING DECK EXPORTED TO: 'Remdesivir_Analogs_Final_Screening_3D.sdf'


,Compound_ID,Valence_Audit,SP3_Atom_Count,SP2_Atom_Count,Net_Charge
0,CID_44468358_iso_1,Pass,17,21,0
1,CID_58059537_iso_1,Pass,15,21,0
2,CID_58059541_iso_1,Pass,16,21,0
3,CID_76314403_iso_1,Pass,18,21,0
4,CID_168297134,Pass,18,21,0
...,...,...,...,...,...
296,CID_168323673_iso_1,Pass,16,21,0
297,CID_169446158,Pass,13,21,0
298,CID_169446183,Pass,17,21,0
299,CID_171338476,Pass,17,21,0


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Keep spreadsheet displays interactive in Colab
data_table.enable_dataframe_formatter()

def run_production_charge_assignment(input_sdf="Remdesivir_Analogs_Final_Screening_3D.sdf", output_sdf="Remdesivir_Analogs_Charged_Master_3D.sdf"):
    """
    Streams the filtered 3D screening deck, calculates Gasteiger partial charges
    for every atom, and writes them out into a parameter-rich docking master file.
    """
    print(f"🧬 Loading final 3D screening deck: '{input_sdf}'...")
    suppl = Chem.SDMolSupplier(input_sdf, removeHs=False)
    if not suppl:
        print(f"❌ Error: Cannot open '{input_sdf}'. Check your file path.")
        return None

    print(f"⚡ Calculating Gasteiger electrostatic profiles across all structures...")
    print(f"📦 Exporting fully charged models to: '{output_sdf}'\n")

    # Initialize centralized writer for the charged 3D screening library
    sdf_writer = Chem.SDWriter(output_sdf)

    charge_diagnostics = []
    success_count = 0

    for mol in suppl:
        if mol is None:
            continue

        comp_id = mol.GetProp("_Name")

        try:
            # 1. Compute Gasteiger Partial Charges
            AllChem.ComputeGasteigerCharges(mol)

            # 2. Extract atomic charge attributes for verification
            atom_charges = []
            for atom in mol.GetAtoms():
                # Fallback to 0.0 if a charge cannot be calculated due to uncommon radicals
                val = atom.GetProp('_GasteigerCharge') if atom.HasProp('_GasteigerCharge') else "0.0"
                atom_charges.append(float(val))

            max_pos = max(atom_charges) if atom_charges else 0.0
            max_neg = min(atom_charges) if atom_charges else 0.0
            total_electronic_sum = sum(atom_charges) if atom_charges else 0.0

            # 3. Save the calculated partial charges inside the SDWriter property tags
            mol.SetProp("Max_Positive_Gasteiger", f"{max_pos:.3f}")
            mol.SetProp("Max_Negative_Gasteiger", f"{max_neg:.3f}")
            mol.SetProp("Gasteiger_Sum", f"{total_electronic_sum:.4f}")

            # Append metadata to report list
            charge_diagnostics.append({
                'Compound_ID': comp_id,
                'Max_Positive_Charge': round(max_pos, 3),
                'Max_Negative_Charge': round(max_neg, 3),
                'Sum_of_Charges': round(total_electronic_sum, 4)
            })

            # Stream the updated structure directly to the output file
            sdf_writer.write(mol)
            success_count += 1

        except Exception as e:
            print(f"⚠️ Warning: Optimization failed for compound {comp_id}. Error: {str(e)}")

    # Safely close the combined writer stream
    sdf_writer.close()

    # Convert out to master frames
    df_charge_report = pd.DataFrame(charge_diagnostics)
    df_charge_report.to_csv("Remdesivir_Analogs_Partial_Charges_Log.csv", index=False)

    print("\n✨ Electrostatic Parameterization Subroutine Complete!")
    print(f"🟩 Total compounds successfully parameterized: {success_count}")
    print(f"📁 Electrostatic data log saved to: 'Remdesivir_Analogs_Partial_Charges_Log.csv'")
    print(f"🚀 CHARGED MASTER SCREENING FILE READY FOR DOCKING: '{output_sdf}'")

    return df_charge_report

# --- Run Partial Charge Pipeline ---
df_charges = run_production_charge_assignment()

# View interactive partial charge summary table
df_charges

🧬 Loading final 3D screening deck: 'Remdesivir_Analogs_Final_Screening_3D.sdf'...
⚡ Calculating Gasteiger electrostatic profiles across all structures...
📦 Exporting fully charged models to: 'Remdesivir_Analogs_Charged_Master_3D.sdf'


✨ Electrostatic Parameterization Subroutine Complete!
🟩 Total compounds successfully parameterized: 301
📁 Electrostatic data log saved to: 'Remdesivir_Analogs_Partial_Charges_Log.csv'
🚀 CHARGED MASTER SCREENING FILE READY FOR DOCKING: 'Remdesivir_Analogs_Charged_Master_3D.sdf'


,Compound_ID,Max_Positive_Charge,Max_Negative_Charge,Sum_of_Charges
0,CID_44468358_iso_1,0.459,-0.462,0.0
1,CID_58059537_iso_1,0.459,-0.468,0.0
2,CID_58059541_iso_1,0.459,-0.465,0.0
3,CID_76314403_iso_1,0.459,-0.462,0.0
4,CID_168297134,0.459,-0.461,0.0
...,...,...,...,...
296,CID_168323673_iso_1,0.459,-0.462,0.0
297,CID_169446158,0.459,-0.480,0.0
298,CID_169446183,0.459,-0.465,0.0
299,CID_171338476,0.459,-0.465,0.0


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Keep table layouts interactive inside the Colab environment
data_table.enable_dataframe_formatter()

def run_production_forcefield_profiling(input_sdf="Remdesivir_Analogs_Charged_Master_3D.sdf", output_sdf="Remdesivir_Analogs_Final_Prepared_3D.sdf"):
    """
    Ingests the charged 3D chemical library, maps force field properties,
    performs geometric relaxation, and tabulates final potential energy metrics.
    """
    print(f"🧬 Loading charged 3D screening deck: '{input_sdf}'...")
    suppl = Chem.SDMolSupplier(input_sdf, removeHs=False)
    if not suppl:
        print(f"❌ Error: Cannot open '{input_sdf}'. Check your file path.")
        return None

    print(f"📐 Initializing MMFF94 parameterization and geometric relaxation loop...")
    print(f"📦 Exporting finalized structural coordinates to: '{output_sdf}'\n")

    # Open the final master file writer for downstream virtual screening applications
    sdf_writer = Chem.SDWriter(output_sdf)

    ff_records = []
    success_count = 0
    failure_count = 0

    for mol in suppl:
        if mol is None:
            continue

        comp_id = mol.GetProp("_Name")

        # 1. Map and validate MMFF94 molecule properties
        mp = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant='MMFF94')
        if mp is None:
            print(f"⚠️ Warning: Molecule {comp_id} contains atom types unsupported by MMFF94.")
            ff_records.append({
                'Compound_ID': comp_id,
                'Forcefield': 'MMFF94',
                'Converged': 'No',
                'Potential_Energy_kcal_mol': 'N/A',
                'Status': 'Unsupported Atom Types'
            })
            failure_count += 1
            continue

        try:
            # 2. Execute final structural relaxation (up to 500 iterations)
            # Returns 0 if successfully converged, 1 if more iterations are required
            convergence_status = AllChem.MMFFOptimizeMolecule(mol, mmffVariant='MMFF94', maxIters=500)

            # 3. Extract the final calculated Potential Energy (kcal/mol)
            ff = AllChem.MMFFGetMoleculeForceField(mol, mp)
            potential_energy = ff.CalcEnergy()

            converged_str = "Yes" if convergence_status == 0 else "No (Max Iterations Reached)"

            # Save data variables as attributes inside the SDF metadata tags
            mol.SetProp("MMFF94_Energy", f"{potential_energy:.2f}")
            mol.SetProp("MMFF94_Converged", converged_str)

            # Record metrics for the tracking table
            ff_records.append({
                'Compound_ID': comp_id,
                'Forcefield': 'MMFF94',
                'Converged': converged_str,
                'Potential_Energy_kcal_mol': round(potential_energy, 2),
                'Status': 'Success'
            })

            # Stream finalized 3D structure to disk workspace
            sdf_writer.write(mol)
            success_count += 1

        except Exception as e:
            ff_records.append({
                'Compound_ID': comp_id,
                'Forcefield': 'MMFF94',
                'Converged': 'No',
                'Potential_Energy_kcal_mol': 'N/A',
                'Status': f'Optimization Error: {str(e)}'
            })
            failure_count += 1

    # Safely close output stream
    sdf_writer.close()

    # Convert out to master dataframes
    df_ff_report = pd.DataFrame(ff_records)
    df_ff_report.to_csv("Remdesivir_Analogs_Energy_Minimization_Log.csv", index=False)

    print("\n✨ Force Field Parameterization Subroutine Complete!")
    print(f"🟩 Successfully optimized and converged ligands: {success_count}")
    print(f"🟥 Failed configurations: {failure_count}")
    print(f"📁 Energy profile log table exported to: 'Remdesivir_Analogs_Energy_Minimization_Log.csv'")
    print(f"🚀 FINAL SCREENING-READY 3D DATASET SAVED AS: '{output_sdf}'")

    return df_ff_report

# --- Execute Force Field Optimization Sequence ---
df_ff = run_production_forcefield_profiling()

# View the interactive energy tracking log sheet
df_ff

🧬 Loading charged 3D screening deck: 'Remdesivir_Analogs_Charged_Master_3D.sdf'...
📐 Initializing MMFF94 parameterization and geometric relaxation loop...
📦 Exporting finalized structural coordinates to: 'Remdesivir_Analogs_Final_Prepared_3D.sdf'


✨ Force Field Parameterization Subroutine Complete!
🟩 Successfully optimized and converged ligands: 301
🟥 Failed configurations: 0
📁 Energy profile log table exported to: 'Remdesivir_Analogs_Energy_Minimization_Log.csv'
🚀 FINAL SCREENING-READY 3D DATASET SAVED AS: 'Remdesivir_Analogs_Final_Prepared_3D.sdf'


,Compound_ID,Forcefield,Converged,Potential_Energy_kcal_mol,Status
0,CID_44468358_iso_1,MMFF94,Yes,-71.99,Success
1,CID_58059537_iso_1,MMFF94,Yes,-61.84,Success
2,CID_58059541_iso_1,MMFF94,Yes,-65.46,Success
3,CID_76314403_iso_1,MMFF94,Yes,-71.92,Success
4,CID_168297134,MMFF94,Yes,-77.76,Success
...,...,...,...,...,...
296,CID_168323673_iso_1,MMFF94,Yes,-76.62,Success
297,CID_169446158,MMFF94,Yes,-95.40,Success
298,CID_169446183,MMFF94,Yes,-79.85,Success
299,CID_171338476,MMFF94,Yes,-85.65,Success


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Keep table layouts interactive inside the Colab environment
data_table.enable_dataframe_formatter()

def run_production_energy_diagnostics(input_sdf="Remdesivir_Analogs_Final_Prepared_3D.sdf", output_sdf="Remdesivir_Analogs_Docking_Ready_3D.sdf"):
    """
    Streams the prepared library, tracks the structural potential energy delta
    across an intense 1000-iteration MMFF94 refinement loop, and writes out
    the finalized, zero-strain docking deck.
    """
    print(f"🧬 Loading prepared 3D screening library: '{input_sdf}'...")
    suppl = Chem.SDMolSupplier(input_sdf, removeHs=False)
    if not suppl:
        print(f"❌ Error: Cannot open '{input_sdf}'. Check your file path configuration.")
        return None

    print(f"⚡ Launching 1000-iteration MMFF94 refinement & delta energy profiling...")
    print(f"📦 Compiling final docking-ready assets to: '{output_sdf}'\n")

    # Open the ultimate master writer for the screen-ready library
    sdf_writer = Chem.SDWriter(output_sdf)

    diagnostic_records = []
    success_count = 0

    for mol in suppl:
        if mol is None:
            continue

        comp_id = mol.GetProp("_Name")
        mp = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant='MMFF94')

        if mp is None:
            continue

        try:
            # 1. Calculate the baseline potential energy (kcal/mol)
            ff_initial = AllChem.MMFFGetMoleculeForceField(mol, mp)
            initial_energy = ff_initial.CalcEnergy()

            # 2. Perform deep iterative minimization to resolve lingering steric strain
            optimization_flag = AllChem.MMFFOptimizeMolecule(mol, mmffVariant='MMFF94', maxIters=1000)

            # 3. Calculate the optimized potential energy (kcal/mol)
            ff_final = AllChem.MMFFGetMoleculeForceField(mol, mp)
            final_energy = ff_final.CalcEnergy()

            # Delta calculation: Represents the amount of geometric strain relieved
            delta_e = initial_energy - final_energy
            converged_status = "Yes" if optimization_flag == 0 else "No"

            # Assign properties inside the production structural file tags
            mol.SetProp("Initial_Energy_kcal_mol", f"{initial_energy:.2f}")
            mol.SetProp("Final_Energy_kcal_mol", f"{final_energy:.2f}")
            mol.SetProp("Delta_Energy_Relieved", f"{delta_e:.2f}")
            mol.SetProp("Deep_Optimization_Converged", converged_status)

            diagnostic_records.append({
                'Compound_ID': comp_id,
                'Initial_Energy': round(initial_energy, 2),
                'Final_Energy': round(final_energy, 2),
                'Delta_E_Relieved': round(delta_e, 2),
                'Minimization_Converged': converged_status
            })

            # Stream structural record out to target master directory
            sdf_writer.write(mol)
            success_count += 1

        except Exception as e:
            print(f"⚠️ Diagnostic skip on compound {comp_id}. Error: {str(e)}")

    # Safely close output stream
    sdf_writer.close()

    # Build evaluation tracker metrics dataframe
    df_diagnostics_report = pd.DataFrame(diagnostic_records)
    df_diagnostics_report.to_csv("Remdesivir_Analogs_Thorough_Minimization_Log.csv", index=False)

    print("\n✨ Delta Convergence Optimization Diagnostics Complete!")
    print(f"🟩 Total compounds checked and structurally verified: {success_count}")
    print(f"📁 Detailed diagnostic log written to: 'Remdesivir_Analogs_Thorough_Minimization_Log.csv'")
    print(f"🚀 VIRTUAL SCREENING PRODUCTION DECK PREPPED AND LOCKED: '{output_sdf}'")

    return df_diagnostics_report

# --- Run Diagnostic Sequencing Grid ---
df_min = run_production_energy_diagnostics()

# View interactive diagnostic summary matrices
df_min

🧬 Loading prepared 3D screening library: 'Remdesivir_Analogs_Final_Prepared_3D.sdf'...
⚡ Launching 1000-iteration MMFF94 refinement & delta energy profiling...
📦 Compiling final docking-ready assets to: 'Remdesivir_Analogs_Docking_Ready_3D.sdf'


✨ Delta Convergence Optimization Diagnostics Complete!
🟩 Total compounds checked and structurally verified: 301
📁 Detailed diagnostic log written to: 'Remdesivir_Analogs_Thorough_Minimization_Log.csv'
🚀 VIRTUAL SCREENING PRODUCTION DECK PREPPED AND LOCKED: 'Remdesivir_Analogs_Docking_Ready_3D.sdf'


,Compound_ID,Initial_Energy,Final_Energy,Delta_E_Relieved,Minimization_Converged
0,CID_44468358_iso_1,-71.99,-71.99,0.0,Yes
1,CID_58059537_iso_1,-61.84,-61.84,0.0,Yes
2,CID_58059541_iso_1,-65.46,-65.46,0.0,Yes
3,CID_76314403_iso_1,-71.92,-71.92,0.0,Yes
4,CID_168297134,-77.76,-77.76,0.0,Yes
...,...,...,...,...,...
296,CID_168323673_iso_1,-76.62,-76.62,0.0,Yes
297,CID_169446158,-95.40,-95.40,0.0,Yes
298,CID_169446183,-79.85,-79.85,0.0,Yes
299,CID_171338476,-85.65,-85.65,0.0,Yes


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import data_table

# Keep table layouts interactive inside the Colab environment
data_table.enable_dataframe_formatter()

def run_production_ensemble_generation(input_sdf="Remdesivir_Analogs_Docking_Ready_3D.sdf", output_sdf="Remdesivir_Analogs_Ensemble_Screening_Deck.sdf", max_confs=50, rmsd_cutoff=1.0):
    """
    Ingests the docking-ready 3D collection, generates an optimized multi-conformer
    ensemble for each analog, filters variations by an RMSD threshold, and writes
    the final virtual screening deck.
    """
    print(f"🧬 Loading verified 3D data collection: '{input_sdf}'...")
    suppl = Chem.SDMolSupplier(input_sdf, removeHs=False)
    if not suppl:
        print(f"❌ Error: Cannot open '{input_sdf}'. Check your file path pathing.")
        return None

    print(f"🔄 Launching multi-threaded rotatable bond ensemble sampler (Max Confs per Entry: {max_confs})...")
    print(f"📦 Exporting pooled screening geometries to: '{output_sdf}'\n")

    # Open our centralized writer for the complete multi-conformer screening deck
    sdf_writer = Chem.SDWriter(output_sdf)

    ensemble_records = []
    total_confs_written = 0
    compound_idx = 0

    for mol in suppl:
        if mol is None:
            continue

        comp_id = mol.GetProp("_Name")
        compound_idx += 1

        try:
            # Generate a distinct geometric ensemble using keyword arguments
            # Setting numThreads=0 tells RDKit to utilize all available CPU cores automatically
            conf_ids = AllChem.EmbedMultipleConfs(
                mol,
                numConfs=max_confs,
                pruneRmsThresh=rmsd_cutoff,
                randomSeed=42,
                useExpTorsionAnglePrefs=True,
                useBasicKnowledge=True,
                numThreads=0
            )

            if conf_ids:
                # Synchronously minimize all generated shapes with MMFF94 across all cores
                AllChem.MMFFOptimizeMoleculeConfs(mol, numThreads=0, mmffVariant='MMFF94')

                generated_count = len(conf_ids)

                # Stream each individual unique conformation into our centralized master screening file
                for conf_id in conf_ids:
                    # Storing conformer index values within the file metadata tags
                    mol.SetProp("Conformer_ID", f"{conf_id}")
                    sdf_writer.write(mol, confId=conf_id)

                total_confs_written += generated_count

                ensemble_records.append({
                    'Compound_ID': comp_id,
                    'Rotatable_Conformers_Found': generated_count,
                    'Status': 'Ensemble Generated Successfully'
                })
            else:
                ensemble_records.append({
                    'Compound_ID': comp_id,
                    'Rotatable_Conformers_Found': 0,
                    'Status': 'Failed: Configuration space too constrained'
                })

        except Exception as e:
            ensemble_records.append({
                'Compound_ID': comp_id,
                'Rotatable_Conformers_Found': 0,
                'Status': f'Failed: Exception error {str(e)}'
            })

    # Safely close output stream
    sdf_writer.close()

    # Format tracking report layout matrix
    df_ensemble_report = pd.DataFrame(ensemble_records)
    df_ensemble_report.to_csv("Remdesivir_Analogs_Ensemble_Generation_Log.csv", index=False)

    print("\n✨ Multi-Conformer Ensemble Processing Complete!")
    print(f"🟩 Total unique lead architectures processed: {compound_idx}")
    print(f"🔥 Aggregate total 3D geometries written to deck: {total_confs_written}")
    print(f"📁 Processing matrix metrics log saved to: 'Remdesivir_Analogs_Ensemble_Generation_Log.csv'")
    print(f"🚀 PRODUCTION SCREENING DECK EXPORTED AND READY FOR H5N1 DOCKING: '{output_sdf}'")

    return df_ensemble_report

# --- Execute Ensemble Pipeline ---
df_full_ensm = run_production_ensemble_generation()

# View the interactive rotatable bond summary logs
df_full_ensm

🧬 Loading verified 3D data collection: 'Remdesivir_Analogs_Docking_Ready_3D.sdf'...
🔄 Launching multi-threaded rotatable bond ensemble sampler (Max Confs per Entry: 50)...
📦 Exporting pooled screening geometries to: 'Remdesivir_Analogs_Ensemble_Screening_Deck.sdf'



[16:14:26] UFFTYPER: Warning: hybridization set to SP3 for atom 7
[16:14:26] UFFTYPER: Unrecognized charge state for atom: 7



✨ Multi-Conformer Ensemble Processing Complete!
🟩 Total unique lead architectures processed: 301
🔥 Aggregate total 3D geometries written to deck: 14226
📁 Processing matrix metrics log saved to: 'Remdesivir_Analogs_Ensemble_Generation_Log.csv'
🚀 PRODUCTION SCREENING DECK EXPORTED AND READY FOR H5N1 DOCKING: 'Remdesivir_Analogs_Ensemble_Screening_Deck.sdf'


,Compound_ID,Rotatable_Conformers_Found,Status
0,CID_44468358_iso_1,46,Ensemble Generated Successfully
1,CID_58059537_iso_1,48,Ensemble Generated Successfully
2,CID_58059541_iso_1,45,Ensemble Generated Successfully
3,CID_76314403_iso_1,49,Ensemble Generated Successfully
4,CID_168297134,48,Ensemble Generated Successfully
...,...,...,...
296,CID_168323673_iso_1,44,Ensemble Generated Successfully
297,CID_169446158,47,Ensemble Generated Successfully
298,CID_169446183,50,Ensemble Generated Successfully
299,CID_171338476,48,Ensemble Generated Successfully
